# The Delta Method

## Load Packages and Extra Functions

The notebook first implements the delta method step-by-step. At the end it also presents a the function `DeltaMethod()` from the (local) `FinEcmt_OLS` module that wraps those calculations.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using DelimitedFiles, Statistics

## Load Data

In [4]:
x = readdlm("Data/TwoIndustries.csv",',',skipstart=1)

(dN,Re,F) = (x[:,1],Float64.(x[:,2:3]),Float64.(x[:,4:end]))   #make sure data is Float

T = size(F,1)              

660

## The Inputs: Parameters and their VCV

In [5]:
Rme = F[:,1]           #equity market excess return

μ   = mean(Rme)                  #estimates of the mean and variance
σ²  = var(Rme,corrected=false)

printblue("mean and variance:")
momNames = ["μ","σ²"]
printmat([μ,σ²];colNames=momNames)

mean and variance:
     0.608
    21.117



The variance-covariance matrix (called $V$) of the point estimates depends on the distribution of the data. With a normal distribution, the form is particularly simple. We use that approximation in the next cell and compare with another approach which estimates $V$ from the moment conditions (see GMM for details).

In [6]:
println("V*100")

V = [σ² 0;                 #VCV assuming N()
     0  2*abs2(σ²)]/T
printmat(V*100;rowNames=momNames,colNames=momNames)

g = hcat(Rme .- μ,(Rme .- μ).^2 .- σ²)   #VCV from moment conditions
V = cov(g;corrected=false)/T
printmat(V*100;rowNames=momNames,colNames=momNames)

V*100
           μ        σ²
μ      3.200     0.000
σ²     0.000   135.133

           μ        σ²
μ      3.200    -7.526
σ²    -7.526   247.285



## The Sharpe Ratio and Its Derivatives

The Sharpe ratio and its derivatives (with respect to the parameters of the
Sharpe ratio) are

$SR(\beta)  =\mu/\sigma,\: \text{where}\: \beta=(\mu,\sigma^2)$

Let $f(\beta)$ denote the Sharpe ratio where $\beta$ is a vector of parameters 
consisting of the mean and the variance ($\mu,\sigma^2$). The derivatives are then

$\frac{\partial f(\beta)}{\partial\beta^{\prime}}  
= \begin{bmatrix}
\frac{1}{\sigma} & \frac{-\mu}{2 \sigma^3}
\end{bmatrix}$

We will refer to the matrix of derivatives as $D$.

In [7]:
"""
    SRFn(μ,σ²)

Calculate the Sharpe ratio from the mean μ and variance σ²

"""
function SRFn(μ,σ²)
  σ  = sqrt(σ²)
  SR = μ/σ
  D = hcat(1/σ, -μ/(2*σ^3))     #Jacobian of SR, 1x2
  return SR, D
end

SRFn

In [8]:
(SR,D) = SRFn(μ,σ²)

printlnPs("Sharpe ratio: ",SR)

printblue("\nDerivatives*100 of Sharpe ratio function wrt:")
printmat(D*100,colNames=momNames)

Sharpe ratio:      0.132

Derivatives*100 of Sharpe ratio function wrt:
         μ        σ²
    21.761    -0.313



## Applying the Delta Method


Recall that if

$\hat{\beta} \sim N(\beta,V),$

then the distribution of the function $f(\hat{\beta})$ is asymptotically

$f(\hat{\beta}) \sim N(f(\beta),DVD')$

where $D$ are the derivatives of $f(\beta)$.

In [9]:
Std_SR = sqrt(only(D*V*D'))  #only() to convert from 1x1 matrix to scalar
tstat = SR/Std_SR

printblue("Results from the delta method:")
printmat([SR Std_SR tstat],colNames=["SR","Std(SR)","t-stat"])

printblue("annualise the SR and Std by multiplying by sqrt(12)")

Results from the delta method:
        SR   Std(SR)    t-stat
     0.132     0.041     3.265

annualise the SR and Std by multiplying by sqrt(12)


## A Function for the Delta Method (extra)

is included below. It uses numerical derivatives from the `FiniteDiff.jl` package. 

To use this, first write a function that takes `(β,x)` as inputs (see `SRFn2(β,x)` below), where `β` is a vector of the parameters and `x` any data needed (for the Sharpe ratio, no data is needed). 

In [10]:
@doc2 DeltaMethod

```julia
DeltaMethod(fn::Function,β,V,x=NaN)
```

Apply the delta method on the function `fn(β,x)`

### Input

  * `fn::Function`:     of the type fn(β,x)
  * `β::Vector`:        with parameters
  * `V::Matrix`:        Cov(β)
  * `x::VecOrMat`:      data (if any is needed)

### Requires

  * `using FiniteDiff: finite_difference_jacobian as jacobian`


In [11]:
using CodeTracking
println(@code_string  DeltaMethod(cos,[1],[1]))

function DeltaMethod(fn::Function,β,V,x=NaN)
    P = jacobian(b->fn(b,x),β)        #numerical Jacobian
    Cov_fn = P*V*P'
    return Cov_fn
end


In [12]:
"""
    SRFn2(β,x)

Function for Sharpe ratio in terms of the vector β. No derivatives
"""
function SRFn2(β,x=NaN)
  (μ,σ²) = β
  σ  = sqrt(σ²)
  SR = μ/σ
  return SR
end;

In [13]:
Var_SR = DeltaMethod(SRFn2,[μ,σ²],V)

printblue("Std of SR from DeltaMethod():")
printmat(sqrt(only(Var_SR)))

printblue("Compare with the earlier results where we used analytical derivatives")

Std of SR from DeltaMethod():
     0.041

Compare with the earlier results where we used analytical derivatives
